# Global Runner - Multi-Region Macro Data Pipeline

**NOTE:** This notebook has been updated to use the refactored `MacroDataPipeline` from `quant_terminal/backend/app/services/macro/engine.py`.

## Setup Instructions
1. Ensure you have a `FRED_API_KEY` in your environment or `.env` file
2. Run all cells to start fetching macro data for all configured regions

In [ ]:
import sys
import os
from pathlib import Path

# Add the quant_terminal backend to Python path
project_root = Path().cwd().parent / 'quant_terminal' / 'backend' / 'app'
sys.path.insert(0, str(project_root))

# Now import from the refactored location
from services.macro.engine import MacroDataPipeline, RegionConfig
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import region configurations
from us_data import get_config as get_us_config
from eu_data import get_config as get_eu_config
from jp_data import get_config as get_jp_config
from cn_data import get_config as get_cn_config
from ca_data import get_config as get_ca_config
from au_data import get_config as get_au_config
from nz_data import get_config as get_nz_config
from ch_data import get_config as get_ch_config

print("✓ All imports successful!")

## Region Configurations
Load all region configurations for multi-region analysis

In [ ]:
# Define all region configurations
configs = [
    get_us_config(),
    get_eu_config(),
    get_jp_config(),
    get_ca_config(),
    get_au_config(),
    get_nz_config(),
    get_cn_config(),
    get_ch_config()
]

print(f"Configured {len(configs)} regions:")
for config in configs:
    print(f"  - {config.region_name}")

## Option 1: Single Run (Fetch All Data Once)

This cell fetches data for all regions once. Ideal for testing or manual data updates.

In [ ]:
import asyncio
from rich.progress import Progress, SpinnerColumn, TextColumn

async def fetch_region_data(config: RegionConfig, api_key: str = None):
    """Fetch data for a single region asynchronously."""
    pipeline = MacroDataPipeline(config=config, api_key=api_key)
    result = await pipeline.fetch_current_data()
    return config.region_name, result

async def fetch_all_regions():
    """Fetch data for all regions in parallel."""
    api_key = os.getenv('FRED_API_KEY')
    
    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        transient=True,
    ) as progress:
        task = progress.add_task("Fetching macro data...", total=None)
        
        tasks = [fetch_region_data(config, api_key) for config in configs]
        results = await asyncio.gather(*tasks)
        
    print("\n✓ Data fetched for all regions:")
    for region_name, data in results:
        print(f"  - {region_name}: {len(data)} indicators")
    
    return dict(results)

# Run the async fetch
all_data = await fetch_all_regions()

## Data Analysis

Explore the fetched data interactively

In [ ]:
import pandas as pd
from rich.table import Table
from rich.console import Console

console = Console()

# Display summary table
table = Table(title="Macro Data Summary")
table.add_column("Region", style="cyan")
table.add_column("GDP", style="green")
table.add_column("Inflation (CPI)", style="yellow")
table.add_column("Interest Rate", style="magenta")
table.add_column("Unemployment", style="red")

for region_name, data in all_data.items():
    if data:
        gdp = data.get('gdp', {}).get('value', 'N/A')
        cpi = data.get('inflation', {}).get('value', 'N/A')
        rate = data.get('rates', {}).get('value', 'N/A')
        unemp = data.get('unemployment', {}).get('value', 'N/A')
        
        table.add_row(
            region_name,
            f"{gdp:.2f}" if isinstance(gdp, (int, float)) else str(gdp),
            f"{cpi:.2f}%" if isinstance(cpi, (int, float)) else str(cpi),
            f"{rate:.2f}%" if isinstance(rate, (int, float)) else str(rate),
            f"{unemp:.2f}%" if isinstance(unemp, (int, float)) else str(unemp)
        )

console.print(table)

## Option 2: Continuous Monitoring (Advanced)

**WARNING:** This will run indefinitely. Use with caution in notebooks.

For production use, consider running `global_runner.py` as a standalone script instead.

In [ ]:
# UNCOMMENT TO ENABLE CONTINUOUS MONITORING
# WARNING: This will block the notebook!

# import threading
# import time
# import logging

# logging.basicConfig(
#     level=logging.INFO,
#     format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
#     handlers=[logging.StreamHandler(sys.stdout)]
# )
# logger = logging.getLogger("GlobalRunner")

# def run_pipeline(config: RegionConfig):
#     """Runs a pipeline continuously for a specific region."""
#     try:
#         pipeline = MacroDataPipeline(config)
#         # This would need to be adapted for the refactored version
#         # The new pipeline doesn't have run_continuously()
#         while True:
#             data = pipeline.fetch_current_data()
#             logger.info(f"Fetched {len(data)} indicators for {config.region_name}")
#             time.sleep(14400)  # 4 hours
#     except Exception as e:
#         logger.error(f"Pipeline for {config.region_name} failed: {e}")

# # Launch threads for each region
# threads = []
# for config in configs:
#     t = threading.Thread(target=run_pipeline, args=(config,), name=f"Thread-{config.region_name}")
#     t.daemon = True
#     t.start()
#     threads.append(t)
#     time.sleep(0.5)

# logger.info("All pipelines running. Press Ctrl+C to stop.")

print("ℹ️  Continuous monitoring disabled. Uncomment code above to enable.")

In [ ]:
"""
Global Runner - Standalone Version

Fetches macroeconomic data for all configured regions using the local standalone engine.
This avoids import path corruption issues by keeping everything within the `macroe` folder.
"""
import asyncio
import sys
import os
import logging
from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv

# Import LOCAL engine (bypassing corrupted services.macro path)
from engine_local import MacroDataPipeline, RegionConfig

# Import LOCAL region configurations
from us_data import get_config as get_us_config
from eu_data import get_config as get_eu_config
from jp_data import get_config as get_jp_config
from cn_data import get_config as get_cn_config
from ca_data import get_config as get_ca_config
from au_data import get_config as get_au_config
from nz_data import get_config as get_nz_config
from ch_data import get_config as get_ch_config

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("GlobalRunner")

# Load environment variables
load_dotenv()

async def fetch_region_data(config: RegionConfig, api_key: str = None) -> tuple[str, Dict[str, Any]]:
    """
    Fetch data for a single region asynchronously.
    """
    try:
        pipeline = MacroDataPipeline(config=config, api_key=api_key)
        data = await pipeline.fetch_current_data()
        logger.info(f"OK - Fetched {len(data)} indicators for {config.region_name}")
        return config.region_name, data
    except Exception as e:
        logger.error(f"ERROR - Failed to fetch data for {config.region_name}: {e}")
        return config.region_name, {}

async def fetch_all_regions(configs: List[RegionConfig]) -> Dict[str, Dict[str, Any]]:
    """
    Fetch data for all regions in parallel.
    """
    api_key = os.getenv('FRED_API_KEY')
    
    if not api_key:
        logger.warning("WARNING: FRED_API_KEY not found. Using mock data.")
    
    logger.info(f"LAUNCHING fetchers for {len(configs)} regions...")
    
    # Fetch all regions in parallel
    tasks = [fetch_region_data(config, api_key) for config in configs]
    results = await asyncio.gather(*tasks)
    
    logger.info(f"DONE - Completed fetching data for {len(results)} regions")
    
    return dict(results)

def main():
    """Main entry point."""
    print("="*60)
    print("GLOBAL RUNNER (STANDALONE MODE)")
    print("="*60)
    
    # Define all region configurations
    configs = [
        get_us_config(),
        get_eu_config(),
        get_jp_config(),
        get_ca_config(),
        get_au_config(),
        get_nz_config(),
        get_cn_config(),
        get_ch_config()
    ]
    
    logger.info(f"Configured regions: {', '.join(c.region_name for c in configs)}")
    
    # Run async fetch
    try:
        all_data = asyncio.run(fetch_all_regions(configs))
        
        # Display summary
        print("\n" + "="*60)
        print("MACRO DATA SUMMARY")
        print("="*60)
        for region, data in all_data.items():
            print(f"{region:15} → {len(data):2} indicators")
        print("="*60)
        
        return all_data
        
    except KeyboardInterrupt:
        logger.info("STOPPING - Shutting down global runner...")
        sys.exit(0)
    except Exception as e:
        logger.error(f"FATAL - Error: {e}", exc_info=True)
        sys.exit(1)

if __name__ == "__main__":
    main()